In [0]:
%python
import pandas as pd
from data import generate_messy_data

In [0]:
%python
df = generate_messy_data()

_Convert pandas DataFrame to Spark DataFrame_

In [0]:
%python
spark_df = spark.createDataFrame(df)
spark_df.write.format('delta').mode('overwrite').option('mergeSchema', 'true').saveAsTable('transform.sql_db.data')
print('DataFrame registered as Delta table: transform.sql_db.data')

In [0]:
-- now when we are data cleaning we usually follow a few steps
-- 1. check for duplicates and remove any
-- 2. standardize data and fix errors
-- 3. Look at null values and see what 
-- 4. remove any columns and rows that are not necessary - few ways


_Clone Raw Table to Staging(Bronze) Layer_

In [0]:
create table transform.sql_db.data_staging
like transform.sql_db.data

-- This only duplicates the schema, not the data

In [0]:
-- This will duplicate the data
INSERT INTO transform.sql_db.data_staging
select * from transform.sql_db.data_raw

-- The reason why we do this is because we want to keep the original data in case we need to go back to it
-- It is not best practice to work on raw data

In [0]:
select * ,REPLACE(email_addr, '#', '@') as email_cleaned from transform.sql_db.data_staging

In [0]:

--Remove Duplicates
--We would use the window function ROW_NUMBER() to identify the duplicates and then remove them
--This is a very common way to remove duplicates
-- But i noticed that the email address has unnecessary characters in it, which will act as distinct values
--So we need to clean the email address first then remove the duplicates 
--We can also use the DISTINCT keyword to remove duplicates
--But this is not as efficient
--It's best practice to include only columns that are unqiue in the partition by clause rather than including all the columns
select *,
  row_number() 
  over (partition by user_id, email_cleaned, sale_date order by user_id) as rn
  FROM (
  select * ,REPLACE(email_addr, '#', '@') as email_cleaned 
  from transform.sql_db.data_staging
)



In [0]:
--Identify Duplicates where rn > 1  using CTE

with dup_record as (
select *,
  row_number() 
  over (partition by user_id, email_cleaned, sale_date order by user_id) as rn
  FROM (
  select * ,REPLACE(email_addr, '#', '@') as email_cleaned 
  from transform.sql_db.data_staging
  )
)
select * from dup_record where rn > 1



In [0]:

--first we have to create a new table with the new column rn using the proper datatype
-- The reason why we have to create a new table is because we can't delete from a table that we are querying from using a simple DELETE FROM statement


CREATE TABLE transform.sql_db.data_staging_new(
  user_id long,
  email_cleaned string,
  region_name string,
  sale_date date,
  base_price double,
  jan_sales long,
  feb_sales long,
  user_rating long,
  rn int
)

In [0]:
-- INSERT THE QUERY INTO THE TABLE
insert into transform.sql_db.data_staging_new
select user_id,email_cleaned,region_name, sale_date ,base_price,jan_sales,feb_sales,user_rating,
  row_number() 
  over (partition by user_id, email_cleaned, sale_date order by user_id) as rn
  FROM (
  select * ,REPLACE(email_addr, '#', '@') as email_cleaned 
  from transform.sql_db.data_staging
  )

In [0]:
-- DELETE THE DUPLICATES
-- Delete rows where rn > 1
delete from transform.sql_db.data_staging_new
where rn > 1

### **2. standardize data and fix errors**

In [0]:
delete from transform.sql_db.data_staging_new;
drop table transform.sql_db.data_staging_new


In [0]:
-- Formatting dates
-- Cleaning text & Removing unwanted characters (we did remove unwanted characters in the previous step for the email columns)
-- Converting data types
-- We can use the cast function to convert data types
-- We can use the trim function to remove unwanted characters
-- We can use the replace function to replace unwanted characters (done)
-- We can use the lower function to convert text to lowercase
-- We can use the upper function to convert text to uppercase
-- We can use the ltrim function to remove leading spaces
-- We can use the rtrim function to remove trailing spaces
-- We can use the lpad function to add leading zeros
-- We can use the rpad function to add trailing zeros
-- Least to clip the max value of the column to 5, any value greater than 5 will be set to 5
-- We can use the GREATEST function to clip the min value of the column to 0, any value less than 0 will be set to 0
-- Use to_date(sale_date, 'dd/MM/yy') to convert the sale_date column to a date format 
--Use d for single digits (1) or dd for (01) -- yyyy(2026)-- yy	(26)-- MM	Must be uppercase. mm stands for minutes!
--COALESCE(sale_id, 0) if sale is null it replaces it with 0
UPDATE transform.sql_db.data_staging_new
set
  sale_date = to_date(sale_date , 'd/MM/yyyy'), -- this is redundant as we already converted the column type previously
  base_price = LEAST(base_price, 1000),
  region_name = lower(region_name),
  user_rating = LEAST(GREATEST(user_rating, 0), 5)

--

In [0]:
update transform.sql_db.data_staging_new
set 
  user_id = TRIM(user_id),
  email_cleaned = TRIM(email_cleaned),
  region_name = TRIM(region_name),
  base_price = TRIM(base_price),
  user_rating = TRIM(user_rating),
  jan_sales = TRIM(jan_sales),
  feb_sales = TRIM(feb_sales)


In [0]:
update transform.sql_db.data_staging_new
set email_cleaned = REPLACE(email_cleaned, '_com', '.com')

In [0]:
select coalesce(user_id,(/) ), email_cleaned from transform.sql_db.data_staging_new

In [0]:
SELECT
    email_cleaned,
    regexp_extract(email_cleaned, 'user_([0-9]+)', 1) AS extracted_id, user_id
FROM transform.sql_db.data_staging_new
WHERE user_id IS NULL;

In [0]:
select distinct user_id, email_cleaned from transform.sql_db.data_staging_new
order by 1
--where email_cleaned = "user_11@example.com"

In [0]:
%python
# 1. Define your table path
table_path = "transform.sql_db.data"

# 2. Get the current column names from Databricks
cols = spark.table(table_path).columns

# 3. Generate the SQL commands
for old_name in cols:
    # Apply your logic: strip, lowercase, and replace spaces
    new_name = old_name.strip().lower().replace(' ', '_')
    
    if old_name != new_name:
        # We use backticks `` in case the old name has spaces
        sql_cmd = f"ALTER TABLE {table_path} RENAME COLUMN `{old_name}` TO {new_name};"
        print(sql_cmd)